# RavenPack Smoke Test

Minimal check: connect to WRDS, pull a sample of RavenPack sentiment data for AAPL, verify the sentiment score formula.

In [6]:
import os
from pathlib import Path
import pandas as pd
import wrds
from dotenv import load_dotenv

load_dotenv(Path().resolve().parent / ".env")

db = wrds.Connection(
    wrds_username=os.environ["WRDS_USERNAME"],
    wrds_password=os.environ["WRDS_PASSWORD"],
)

def pg_sql(db, sql):
    """Query WRDS via psycopg2 directly (bypasses SQLAlchemy 2.x bug in wrds 3.x)."""
    cur = db.connection.connection.cursor()
    cur.execute(sql)
    df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.close()
    return df

print("Connected to WRDS")

Loading library list...
Done
Connected to WRDS


In [7]:
# Look up AAPL's RavenPack entity ID
mapping = pg_sql(db, """
    SELECT DISTINCT rp_entity_id, entity_name, ticker
    FROM ravenpack_common.wrds_rpa_company_mappings
    WHERE ticker = 'AAPL'
""")
print(mapping)

  rp_entity_id entity_name ticker
0       D8442A  Apple Inc.   AAPL


In [8]:
rp_entity_id = mapping["rp_entity_id"].iloc[0]

# Pull 50 rows from Q1 2007 — fast sanity check
df = pg_sql(db, f"""
    SELECT timestamp_utc, rp_story_id, relevance, event_sentiment_score,
           headline, event_text
    FROM ravenpack_dj.rpa_djpr_equities_2007
    WHERE rp_entity_id = '{rp_entity_id}'
      AND rpa_date_utc BETWEEN '2007-01-01' AND '2007-03-31'
    ORDER BY timestamp_utc
    LIMIT 50
""")

print(f"{len(df)} rows")
df.head()

50 rows


,timestamp_utc,rp_story_id,relevance,event_sentiment_score,headline,event_text
0,2007-01-01 07:00:00,B8005F2DB2D1005195D5E949B330FBAB,96.0,0.68,Mossberg Report: Thinking Outside The Pod --- ...,Apple's iPod music players are wildly popular
1,2007-01-01 07:00:00,1E0EB77B3148E72B10753839AC85B3A4,3.0,NaN,REVIEW & PREVIEW ---- Edited by Robin Goldwyn ...,None
2,2007-01-01 07:00:00,8FF149F017D4C8BA9A46E337887CF40B,2.0,NaN,Stockscreen: Keeping Score --- A year's worth ...,None
3,2007-01-01 07:00:00,29225E52EAD1EFD94E9756723B710459,35.0,NaN,TECHNOLOGY WEEK --- Tech Trader: The Memory Gl...,None
4,2007-01-01 07:00:00,D4146E8F818EF45978F464BE03CBBE5F,3.0,NaN,Emerging and Converging ---- By Sandra Ward,None


In [9]:
df["relevance_score"]       = pd.to_numeric(df["relevance"], errors="coerce") / 100
df["event_sentiment_score"] = pd.to_numeric(df["event_sentiment_score"], errors="coerce")
df["sentiment_score"]       = df["relevance_score"] * df["event_sentiment_score"]

print(f"Rows with sentiment_score : {df['sentiment_score'].notna().sum()} / {len(df)}")
print(f"Score range               : [{df['sentiment_score'].min():.3f}, {df['sentiment_score'].max():.3f}]")

# Show headline + event_text side by side for rows that have a sentiment score
sample = (
    df[["timestamp_utc", "headline", "event_text", "relevance_score", "event_sentiment_score", "sentiment_score"]]
    .dropna(subset=["sentiment_score"])
    .head(10)
)

# Print event_text in full so it's readable
pd.set_option("display.max_colwidth", None)
sample

Rows with sentiment_score : 20 / 50
Score range               : [-0.584, 0.653]


,timestamp_utc,headline,event_text,relevance_score,event_sentiment_score,sentiment_score
0,2007-01-01 07:00:00.000,Mossberg Report: Thinking Outside The Pod --- A host of alternative digital music sources are looking to take a bite out of Apple ---- By Walter S. Mossberg,Apple's iPod music players are wildly popular,0.96,0.68,0.6528
6,2007-01-01 20:45:54.000,Overnight Markets Report,Apple Computer and Goodyear Tire surged,0.01,0.47,0.0047
9,2007-01-01 22:09:19.992,ASIAN MORNING BRIEFING: US Shares Slip As Oil Gains,Apple Computer and Goodyear Tire surged,0.01,0.47,0.0047
10,2007-01-02 00:46:03.054,WSJ(1/2) Apple Probe Spotlights Two Former Executives,Mr. Anderson to join Apple's board,0.99,0.47,0.4653
11,2007-01-02 00:46:03.054,WSJ(1/2) Apple Probe Spotlights Two Former Executives,Mr. Anderson resigned from Apple's board,0.99,-0.51,-0.5049
13,2007-01-02 02:00:55.528,WSJ(1/2) Year-End Review Of Markets & Finance -6-,U.S. prosecutors are investigating Apple's,0.05,-0.47,-0.0235
14,2007-01-02 02:16:04.511,"WSJ(1/2) Year-End Review Of Markets & Finance 2006: Telecoms, Energy, China Bets Yield Big Returns",Apple's shares rose 18% as,0.03,0.30,0.0090
17,2007-01-02 07:00:00.000,Year-End Review of Markets & Finance: Emerging-markets stocks are among star performers in the U.S. ---- By Scott Patterson,Apple's shares rose 18% as,0.04,0.30,0.0120
18,2007-01-02 07:00:00.000,"Corporate News: Apple clears Jobs of wrongdoing --- Options probe finds CEO recommended 'favorable' dates ---- By Nick Wingfield, Steve Stecklow and Charles Forelle",Apple restated its financial results between 1998 and 2006,0.99,-0.59,-0.5841
19,2007-01-02 07:00:00.000,Year-End Review of Markets & Finance 2006: Emerging-markets stocks are among star performers in the U.S. ---- By Scott Patterson,Apple's shares rose 18% as,0.04,0.30,0.0120


In [10]:
db.close()
print("Done.")

Done.
